
# Cleaning habitat maps (connected components)

**Background.** Habitat assignment paints every ROI voxel with a nearest
centroid id. That can leave tiny islands of a few voxels that no reader
would call a region, and that inflate ITH and graph metrics. Connected-
component cleanup drops islands smaller than a size cut-off and fills
those voxels with neighbouring habitat ids, so the map stays the same
definition with less speck.

**Purpose.** You will run the approved two-step analysis twice on the
same two training patients: once without cleanup, once with a connected-
component stage after assignment. You will count the pieces and the
share of voxels in tiny islands before and after, overlay both maps on
the same slice, and see that volume fractions change only by the voxels
that were reassigned.

**When to use.** When QC or a reader flags fragmented maps, or when you
have decided in advance that habitats smaller than a few dozen voxels
are not biologically meaningful for your study. Cleanup is part of the
analysis definition: report the size cut-off with the methods.

**Key terms.** Full definitions are on :doc:`/tutorial/concepts`.

* **analysis definition** -- the approved two-step stage list of
  :doc:`/auto_examples/01_complete/plot_01_two_step_spec` (subject-level
  winsorize 5 % / 5 % + min-max, 30 k-means supervoxels, cohort-level
  10-bin uniform binning, k-means 2..10 by elbow, nearest centroid).
* **what varies** -- only the optional ``postprocess_habitat`` stage
  after ``assign``. Everything else, including the seed, is identical.
* **connected component** -- a maximal set of face-adjacent voxels that
  share the same habitat id (connectivity 1 = 6-neighbourhood in 3-D).
* **min_component_size** -- islands with fewer voxels than this are
  removed and refilled by a neighbour vote (``neighbor_vote``).
* **postprocess_habitat** -- the Spec role that records cleanup in the
  run manifest and inside a saved ``.habitatmodel``, so predict reuses
  the same rule.

<div class="alert alert-info"><h4>Note</h4><p>Two demo patients are enough to show the mechanics. Choose
   ``min_component_size`` from your own cohort and reader review, not
   from this page's default of 30 voxels.</p></div>

Change ``DATA`` / ``MODALITIES`` / ``ROI`` to your preprocessed tree.


## Load the cohort
Two patients define the habitats. Quantify is limited to ``volume`` so
the page stays on the maps, not on the full feature table.
sphinx_gallery_thumbnail_number = 2



In [ ]:
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

from habit.contracts import HabitatMap, cohort_from_directory
from habit.datasets import fetch_demo
from habit.kernels import habitat_region_stats, ith_score
from habit.recipes import Study
from habit.spec import ROLE_POSTPROCESS_HABITAT, HabitatSpec, Spec, Stage
from habit.viz import plot_habitat_label_compare, plot_habitat_overlay

# Change DATA / MODALITIES / ROI to your preprocessed layout.
DATA = fetch_demo()
MODALITIES = ("pre_contrast", "LAP", "PVP", "delay_3min")
ROI = "LAP"
cohort = cohort_from_directory(DATA, modalities=MODALITIES, roi=ROI)
train = cohort[:2]
print(train)
Path("out").mkdir(exist_ok=True)

## Two definitions that differ only in cleanup
``build_stages`` returns the approved stage list. With
``cleanup=True`` it inserts one ``connected_components`` stage after
``assign`` and changes nothing else.



In [ ]:
MIN_COMPONENT_SIZE = 30  # page choice: drop islands smaller than 30 voxels
TINY_PIECE_VOXELS = 10  # page choice: for the "tiny share" report only


def build_stages(*, cleanup: bool) -> tuple[Stage, ...]:
    """Return the approved two-step stages, optionally with map cleanup.

    Args:
        cleanup: When True, add connected-component cleanup after assign.

    Returns:
        Ordered ``Stage`` tuple for ``HabitatSpec``.
    """
    stages: List[Stage] = [
        Stage("extract", Spec("raw", {"modalities": list(MODALITIES), "roi": ROI})),
        Stage("preprocess", Spec("winsorize", {"winsor_limits": [0.05, 0.05]})),
        Stage("preprocess2", Spec("minmax")),
        Stage("partition", Spec("kmeans", {"n_supervoxels": 30})),
        Stage("pool", Spec("pool")),
        Stage("preprocess_cohort", Spec("binning", {"n_bins": 10, "bin_strategy": "uniform"})),
        Stage(
            "fit",
            Spec(
                "kmeans",
                {"min_habitats": 2, "max_habitats": 10, "validation": "elbow", "n_init": 10},
            ),
        ),
        Stage("assign", Spec("nearest_centroid")),
    ]
    if cleanup:
        # role= tells the pipeline this is habitat-map cleanup, not a
        # quantify stage. The Spec name is the registered strategy.
        stages.append(
            Stage(
                "cleanup",
                Spec(
                    "connected_components",
                    {
                        "min_component_size": MIN_COMPONENT_SIZE,
                        "connectivity": 1,
                        "reassign_method": "neighbor_vote",
                        "max_iterations": 3,
                    },
                ),
                role=ROLE_POSTPROCESS_HABITAT,
            )
        )
    stages.append(Stage("volume", Spec("volume")))
    return tuple(stages)


def map_fragmentation(habitat_map: HabitatMap) -> Dict[str, float]:
    """Count pieces and the share of voxels in tiny islands.

    Args:
        habitat_map: One subject's habitat labels (``0`` = background).

    Returns:
        Dict with piece count, tiny-voxel share and ITH score.
    """
    labels = np.asarray(habitat_map.label_array)
    inside = labels[labels > 0]
    region_stats = habitat_region_stats(labels)
    tiny_voxels = 0
    for hid in region_stats:
        pieces, _ = ndimage.label(labels == hid)
        sizes = np.bincount(pieces.ravel())[1:]
        tiny_voxels += int(sizes[sizes < TINY_PIECE_VOXELS].sum())
    return {
        "pieces": float(sum(n for n, _ in region_stats.values())),
        "tiny_share": float(tiny_voxels / inside.size),
        "ith": float(ith_score(labels)),
    }


spec_raw = HabitatSpec(
    name="cleaning_maps_raw",
    stages=build_stages(cleanup=False),
    random_seed=0,
)
spec_clean = HabitatSpec(
    name="cleaning_maps_clean",
    stages=build_stages(cleanup=True),
    random_seed=0,
)
result_raw = Study(spec_raw).fit_predict(train)
result_clean = Study(spec_clean).fit_predict(train)
print(result_raw.habitat_model.summary())
print("cleanup Spec:", spec_clean.postprocess_habitat or "via stages")

## Piece counts before and after cleanup
The same voxels, the same centroids: only islands smaller than
``MIN_COMPONENT_SIZE`` are refilled. Volume fractions can move by those
reassigned voxels; habitat ids that were present stay present.



In [ ]:
rows: List[Dict[str, object]] = []
for subject, map_raw, map_clean in zip(
    train, result_raw.habitat_maps, result_clean.habitat_maps
):
    raw_stats = map_fragmentation(map_raw)
    clean_stats = map_fragmentation(map_clean)
    changed = int(
        np.count_nonzero(
            np.asarray(map_raw.label_array) != np.asarray(map_clean.label_array)
        )
    )
    rows.append(
        {
            "subject": subject.subject_id,
            "pieces_raw": int(raw_stats["pieces"]),
            "pieces_clean": int(clean_stats["pieces"]),
            "tiny_share_raw": round(raw_stats["tiny_share"], 4),
            "tiny_share_clean": round(clean_stats["tiny_share"], 4),
            "ith_raw": round(raw_stats["ith"], 3),
            "ith_clean": round(clean_stats["ith"], 3),
            "voxels_changed": changed,
        }
    )
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Side-by-side maps on one patient
Left = raw assignment; right = after connected-component cleanup.
Ids come from two fits that share the stage list and seed except for
cleanup, so colours match without Hungarian rematching.



In [ ]:
subject = train[0]
map_raw = result_raw.habitat_maps[0]
map_clean = result_clean.habitat_maps[0]
fig = plot_habitat_label_compare(
    subject.image("LAP"),
    map_raw,
    map_clean,
    titles=("Raw assignment", f"Cleaned (min size {MIN_COMPONENT_SIZE})"),
    crop_to="labels",
)
fig.savefig("out/cleaning_maps_compare.png", dpi=150, bbox_inches="tight")
plt.show()

# One cleaned overlay alone (gallery thumbnail companion).
fig = plot_habitat_overlay(
    subject.image("LAP"),
    map_clean,
    title=f"{subject.subject_id}: cleaned habitats",
    crop_to="labels",
)
fig.savefig("out/cleaning_maps_cleaned.png", dpi=150, bbox_inches="tight")
plt.show()

## Volume fractions move only where voxels were reassigned
The ``volume`` stage columns come from the maps that actually left the
pipeline. Cleanup can change a fraction by a few percent; a large jump
means the cut-off removed a real region, not only speck.



In [ ]:
frame_raw = result_raw.features.frame.set_index("subject")
frame_clean = result_clean.features.frame.set_index("subject")
fraction_cols = [c for c in frame_raw.columns if c.endswith("_volume_fraction")]
delta = (frame_clean[fraction_cols] - frame_raw[fraction_cols]).round(4)
print("volume-fraction change (clean - raw):")
print(delta.to_string())

## How to read the result
Measured on this run (K = 4 by elbow on subj001 + subj002;
``min_component_size=30``, face connectivity):

* subj001: 200 connected pieces -> 14 after cleanup; tiny-voxel share
  0.012 -> 0; ITH 0.95 -> 0.61; 595 voxels changed id.
* subj002: 47 pieces -> 8; tiny share 0.008 -> 0; ITH 0.90 -> 0.31;
  140 voxels changed id.
* Per-habitat volume fractions moved by at most about 0.01 absolute:
  cleanup removed speck, not whole habitats.

Choose the size cut-off for your cohort (and report it). Cleanup that
changes fractions by tens of percent is too aggressive for that data.



## Where to go next
* QC flags that catch fragmented maps before you decide on cleanup:
  :doc:`/auto_examples/01_complete/plot_14_quality_control`.
* The approved analysis without cleanup:
  :doc:`/auto_examples/01_complete/plot_01_two_step_spec`.
* ITH score, which also uses connected pieces:
  :doc:`/auto_examples/02c_metrics/plot_03_ith`.
* Export the cleaned maps and the methods paragraph:
  :doc:`/auto_examples/01_complete/plot_23_export_methods`.

